In [1]:
# %pip install bt pandas numpy matplotlib polars mlforecast

In [2]:
# %pip install hyperopt

In [3]:
# %pip install numpy==1.25.2 pandas==2.0.3 gym_mtsim

In [4]:
import numpy as np
print(np.__version__)
import pandas as pd
print(pd.__version__)

1.25.2
2.0.3


In [ ]:
# filepath = "C:/Users/WilliamFetzner/Downloads/"
# csv_filename = "EURJPY_08_09_23_to_08_10_24.csv"
# import os
# tickdata_test = pd.read_csv(f'{filepath}prepped_{csv_filename}')
# # Calculate split points
# file_length = len(tickdata_test)
# split_points = [file_length // 3, 2 * (file_length // 3)]

# # Split the DataFrame
# df1 = tickdata_test.iloc[:split_points[0]]
# # df2 = tickdata_test.iloc[split_points[0]:split_points[1]]
# # df3 = tickdata_test.iloc[split_points[1]:]

# # Save the split DataFrames
# output_dir = f'{filepath}/prepped_data/'
# os.makedirs(output_dir, exist_ok=True)  # Ensure the directory exists

# df1.to_csv(f'{output_dir}prepped_{csv_filename}(1).csv')
# # df2.to_csv(f'{output_dir}prepped_{csv_filename}(2).csv')
# # df3.to_csv(f'{output_dir}prepped_{csv_filename}(3).csv')

# # Create the csvs list with new filepaths
# csvs = [
#     f'{output_dir}prepped_{csv_filename}(1).csv',
#     f'{output_dir}prepped_{csv_filename}(2).csv',
#     f'{output_dir}prepped_{csv_filename}(3).csv',
# ]

KeyboardInterrupt: 

In [ ]:
import pickle
import sys
sys.path.append("C:/Users/WilliamFetzner/Documents/Trading/")
from gym_mtsim_forked.gym_mtsim.data import FOREX_DATA_PATH, FOREX_DATA_PATH_15MIN
import pytz
from datetime import datetime, timedelta, time
from gym_mtsim import MtSimulator, OrderType
from hyperopt import hp, fmin, tpe, STATUS_OK, Trials, STATUS_FAIL
import matplotlib.pyplot as plt
import mplfinance as mpf

c:\Users\WilliamFetzner\Documents\Trading\Indicator_backtesting\Indicator_search\indicatorSearch\Lib\site-packages\gymnasium\envs\registration.py:642: UserWarning: WARN: Overriding environment forex-hedge-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
c:\Users\WilliamFetzner\Documents\Trading\Indicator_backtesting\Indicator_search\indicatorSearch\Lib\site-packages\gymnasium\envs\registration.py:642: UserWarning: WARN: Overriding environment forex-unhedge-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
c:\Users\WilliamFetzner\Documents\Trading\Indicator_backtesting\Indicator_search\indicatorSearch\Lib\site-packages\gymnasium\envs\registration.py:642: UserWarning: WARN: Overriding environment stocks-hedge-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
c:\Users\WilliamFetzner\Documents\Trading\Indicator_backtesting\Indicator_search\i

In [6]:
path_to_trading_folder = 'C:/Users/WilliamFetzner/Documents/Trading/'
# data_path = 'EURUSD_full_tickstory_data_15_min.csv'
data_path = 'EURUSD_full_tickstory_data_hourly.csv'

In [25]:
# full_15min = pd.read_csv(f'{path_to_trading_folder}data_files/EURUSD_full_tickstory_data_15_min.csv').rename(
#     columns={'Unnamed: 0': 'Time'}
# ).set_index('Time')
# full_15min.index = pd.to_datetime(full_15min.index).tz_localize('UTC', ambiguous='infer')
# full_15min

In [27]:
with open(FOREX_DATA_PATH_15MIN, 'rb') as f:
    symbols_1hr = pickle.load(f)
# convert symbols_1hr to a pd.dataframe
symbols_1hr[1]['EURUSD'].index = pd.to_datetime(symbols_1hr[1]['EURUSD'].index)
full_data = symbols_1hr[1]['EURUSD']
# symbols_1hr[1]['EURUSD'] = full_15min
# with open(FOREX_DATA_PATH_15MIN, 'wb') as f:
#     pickle.dump(symbols_1hr, f)
full_data = full_data[full_data.index.year >= 2024]
full_data

,Open,High,Low,Close
Time,,,,
2024-01-02 00:00:00+00:00,1.10427,1.10433,1.10423,1.10431
2024-01-02 00:15:00+00:00,1.10430,1.10434,1.10420,1.10430
2024-01-02 00:30:00+00:00,1.10431,1.10437,1.10431,1.10433
2024-01-02 00:45:00+00:00,1.10426,1.10447,1.10423,1.10438
2024-01-02 01:00:00+00:00,1.10446,1.10446,1.10383,1.10383
...,...,...,...,...
2024-08-09 22:45:00+00:00,1.09183,1.09189,1.09179,1.09187
2024-08-09 23:00:00+00:00,1.09186,1.09194,1.09167,1.09168
2024-08-09 23:15:00+00:00,1.09168,1.09179,1.09167,1.09175


In [ ]:
class TradingGeekStrategy:
    def __init__(self, df, timeframe='15min', high_timeframe='4H'):
        """
        Initialize the trading strategy
        
        Parameters:
        -----------
        df : pandas.DataFrame
            OHLC dataframe with columns: Open, High, Low, Close
        timeframe : str, optional
            Trading timeframe (default: 15min)
        high_timeframe : str, optional
            Higher timeframe for trend analysis (default: 4H)
        """
        
        self.df = df.copy()
        self.timeframe = timeframe
        self.high_timeframe = high_timeframe
        
        # Prepare dataframes for different timeframes
        self.df_high = self._resample_timeframe(self.df, self.high_timeframe)
        self.df_entry = self._resample_timeframe(self.df, self.timeframe)
        
    def _resample_timeframe(self, df, timeframe):
        """
        Resample dataframe to specified timeframe
        
        Parameters:
        -----------
        df : pandas.DataFrame
            Original OHLC dataframe
        timeframe : str
            Timeframe to resample to (e.g., '4H', '15min')
        
        Returns:
        --------
        pandas.DataFrame
            Resampled dataframe
        """
        return df.resample(timeframe).agg({
            'Open': 'first', 
            'High': 'max', 
            'Low': 'min', 
            'Close': 'last'
        }).dropna()
    
    def identify_supply_demand_zones(self, df, lookback=10, threshold=0.2):
        """
        Identify supply and demand zones based on price action
        
        Parameters:
        -----------
        df : pandas.DataFrame
            Input dataframe
        lookback : int, optional
            Number of periods to look back (default: 10)
        threshold : float, optional
            Percentage threshold for zone identification (default: 0.2)
        
        Returns:
        --------
        dict
            Dictionary of supply and demand zones
        """
        # Identify demand zones (strong bounces from lower levels)
        demand_zones = []
        for i in range(lookback, len(df)):
            recent_lows = df['Low'].iloc[i-lookback:i]
            if df['Low'].iloc[i] > recent_lows.min() * (1 + threshold):
                demand_zones.append({
                    'start': recent_lows.min(),
                    'end': recent_lows.min() * (1 + threshold),
                    'index': df.index[i]
                })
        
        # Identify supply zones (strong rejections from higher levels)
        supply_zones = []
        for i in range(lookback, len(df)):
            recent_highs = df['High'].iloc[i-lookback:i]
            if df['High'].iloc[i] < recent_highs.max() * (1 - threshold):
                supply_zones.append({
                    'start': recent_highs.max() * (1 - threshold),
                    'end': recent_highs.max(),
                    'index': df.index[i]
                })
        
        return {
            'demand_zones': demand_zones,
            'supply_zones': supply_zones
        }
    
    def calculate_supertrend(self, df, period=10, multiplier=3):
        """
        Calculate the Supertrend indicator
        
        Parameters:
        -----------
        df : pandas.DataFrame
            Input dataframe
        period : int, optional
            Period for ATR calculation (default: 10)
        multiplier : float, optional
            Multiplier for ATR (default: 3)
        
        Returns:
        --------
        pandas.DataFrame
            Dataframe with Supertrend indicator
        """
        # Calculate Average True Range (ATR)
        high = df['High']
        low = df['Low']
        close = df['Close']
        
        # True Range
        df['tr'] = np.maximum(
            high - low, 
            np.maximum(abs(high - close.shift(1)), abs(low - close.shift(1)))
        )
        
        # Average True Range
        df['atr'] = df['tr'].rolling(window=period).mean()
        
        # Basic Upperband = High + (Multiplier * ATR)
        # Basic Lowerband = Low - (Multiplier * ATR)
        df['basic_upperband'] = high + (multiplier * df['atr'])
        df['basic_lowerband'] = low - (multiplier * df['atr'])
        
        # Initialize Supertrend columns
        df['final_upperband'] = df['basic_upperband']
        df['final_lowerband'] = df['basic_lowerband']
        df['supertrend'] = np.nan
        
        # Calculate Supertrend
        for i in range(period, len(df)):
            curr = df.iloc[i]
            prev = df.iloc[i-1]
            
            # Adjust Final Upperband
            if curr['Close'] > prev['final_upperband']:
                df.loc[df.index[i], 'final_upperband'] = min(curr['basic_upperband'], prev['final_upperband'])
            
            # Adjust Final Lowerband
            if curr['Close'] < prev['final_lowerband']:
                df.loc[df.index[i], 'final_lowerband'] = max(curr['basic_lowerband'], prev['final_lowerband'])
            
            # Determine Supertrend
            if curr['Close'] <= curr['final_upperband']:
                df.loc[df.index[i], 'supertrend'] = curr['final_upperband']
            else:
                df.loc[df.index[i], 'supertrend'] = curr['final_lowerband']
        
        # Supertrend Direction
        df['supertrend_direction'] = np.where(
            df['Close'] > df['supertrend'], 1, 
            np.where(df['Close'] < df['supertrend'], -1, 0)
        )
        
        return df
    
    def detect_market_structure(self, df, period=10, multiplier=3):
        """
        Detect market structure using Supertrend indicator
        
        Parameters:
        -----------
        df : pandas.DataFrame
            Input dataframe
        period : int, optional
            Period for Supertrend calculation (default: 10)
        multiplier : float, optional
            Multiplier for Supertrend (default: 3)
        
        Returns:
        --------
        str
            Market structure (uptrend, downtrend, range)
        """
        # Calculate Supertrend
        df_supertrend = self.calculate_supertrend(df, period, multiplier)
        
        # Get last few Supertrend directions
        recent_directions = df_supertrend['supertrend_direction'].tail(5)
        
        # Determine trend
        if (recent_directions == 1).sum() >= 3:
            return 'uptrend'
        elif (recent_directions == -1).sum() >= 3:
            return 'downtrend'
        else:
            return 'range'
    
    def detect_liquidity_sweeps(self, df, window=5):
        """
        Detect potential liquidity sweeps
        
        Parameters:
        -----------
        df : pandas.DataFrame
            Input dataframe
        window : int, optional
            Window size for sweep detection (default: 5)
        
        Returns:
        --------
        list
            List of detected liquidity sweeps
        """
        sweeps = []
        for i in range(window, len(df)):
            # Look for potential stop loss triggers
            recent_low = df['Low'].iloc[i-window:i].min()
            recent_high = df['High'].iloc[i-window:i].max()
            
            # Bullish sweep (price drops below recent low)
            if df['Low'].iloc[i] < recent_low:
                sweeps.append({
                    'type': 'bullish_sweep',
                    'price': df['Low'].iloc[i],
                    'index': df.index[i]
                })
            
            # Bearish sweep (price rises above recent high)
            if df['High'].iloc[i] > recent_high:
                sweeps.append({
                    'type': 'bearish_sweep',
                    'price': df['High'].iloc[i],
                    'index': df.index[i]
                })
        
        return sweeps
    
    def generate_trade_signals(self):
        """
        Generate trade signals based on the Trading Geek's strategy
        
        Returns:
        --------
        pandas.DataFrame
            Dataframe with trade signals
        """
        # Initialize signal columns
        self.df_entry['signal'] = 0
        
        # Analyze higher timeframe market structure
        high_tf_structure = self.detect_market_structure(self.df_high)
        
        # Identify supply and demand zones
        zones = self.identify_supply_demand_zones(self.df_high)
        
        # Detect liquidity sweeps
        liquidity_sweeps = self.detect_liquidity_sweeps(self.df_entry)
        
        # Generate trade signals
        for i in range(1, len(self.df_entry)):
            current_price = self.df_entry['Close'].iloc[i]
            prev_price = self.df_entry['Close'].iloc[i-1]
            
            # Long entry conditions
            if high_tf_structure == 'uptrend':
                # Check for demand zone proximity
                demand_zone_entry = any(
                    zone['start'] <= current_price <= zone['end'] 
                    for zone in zones['demand_zones']
                )
                
                # Check for bullish liquidity sweep
                bullish_sweep = any(
                    sweep['type'] == 'bullish_sweep' 
                    for sweep in liquidity_sweeps 
                    if sweep['index'] == self.df_entry.index[i]
                )
                
                # Long entry signal
                if demand_zone_entry and bullish_sweep:
                    self.df_entry.loc[self.df_entry.index[i], 'signal'] = 1
            
            # Short entry conditions
            elif high_tf_structure == 'downtrend':
                # Check for supply zone proximity
                supply_zone_entry = any(
                    zone['start'] <= current_price <= zone['end'] 
                    for zone in zones['supply_zones']
                )
                
                # Check for bearish liquidity sweep
                bearish_sweep = any(
                    sweep['type'] == 'bearish_sweep' 
                    for sweep in liquidity_sweeps 
                    if sweep['index'] == self.df_entry.index[i]
                )
                
                # Short entry signal
                if supply_zone_entry and bearish_sweep:
                    self.df_entry.loc[self.df_entry.index[i], 'signal'] = -1
        
        return self.df_entry
    
    def calculate_trade_management(self):
        """
        Calculate trade management parameters
        
        Returns:
        --------
        pandas.DataFrame
            Dataframe with trade management details
        """
        # Generate initial signals
        signals = self.generate_trade_signals()
        
        # Initialize trade management columns
        signals['stop_loss'] = np.nan
        signals['take_profit'] = np.nan
        signals['risk_reward_ratio'] = np.nan
        
        # Calculate stop loss and take profit
        for i in range(len(signals)):
            if signals['signal'].iloc[i] == 1:  # Long trade
                # Stop loss below recent demand zone
                stop_loss = signals['Low'].iloc[i-10:i].min()
                
                # Take profit at supply zone or fibonacci extension
                take_profit = signals['High'].iloc[i-10:i].max() * 1.618  # Golden ratio
                
                signals.loc[signals.index[i], 'stop_loss'] = stop_loss
                signals.loc[signals.index[i], 'take_profit'] = take_profit
                signals.loc[signals.index[i], 'risk_reward_ratio'] = \
                    (take_profit - signals['Close'].iloc[i]) / \
                    (signals['Close'].iloc[i] - stop_loss)
            
            elif signals['signal'].iloc[i] == -1:  # Short trade
                # Stop loss above recent supply zone
                stop_loss = signals['High'].iloc[i-10:i].max()
                
                # Take profit at demand zone or fibonacci extension
                take_profit = signals['Low'].iloc[i-10:i].min() * 0.618  # Golden ratio
                
                signals.loc[signals.index[i], 'stop_loss'] = stop_loss
                signals.loc[signals.index[i], 'take_profit'] = take_profit
                signals.loc[signals.index[i], 'risk_reward_ratio'] = \
                    (signals['Close'].iloc[i] - take_profit) / \
                    (stop_loss - signals['Close'].iloc[i])
        
        return signals
    
    def visualize_trades(self):
        """
        Visualize trade signals and important levels
        """
        # Get trade management details
        trade_signals = self.calculate_trade_management()
        
        # Prepare mplfinance style
        mc = mpf.make_marketcolors(
            up='g', down='r', 
            edge='inherit', 
            wick={'up':'g', 'down':'r'}
        )
        s = mpf.make_mpf_style(marketcolors=mc)
        
        # Prepare additional plots for signals
        trades = trade_signals[trade_signals['signal'] != 0]
        buy_signals = trades[trades['signal'] == 1]
        sell_signals = trades[trades['signal'] == -1]
        
        # Create additional plot for trade markers
        ap = [
            mpf.make_addplot(buy_signals['Close'], type='scatter', markersize=100, marker='^', color='g'),
            mpf.make_addplot(sell_signals['Close'], type='scatter', markersize=100, marker='v', color='r')
        ]
        
        # Plot
        mpf.plot(
            self.df_entry, 
            type='candle', 
            style=s, 
            title='Trading Geek Strategy - Trade Signals', 
            ylabel='Price',
            addplot=ap
        )

# Example usage
# strategy = TradingGeekStrategy(df)
# trade_signals = strategy.calculate_trade_management()
# strategy.visualize_trades()

In [9]:
def find_seconds_to_next_news(df, news_counts):
    # Ensure both DataFrames have their time columns as datetime
    df['Time'] = pd.to_datetime(df['Time'])
    news_counts['datetime'] = pd.to_datetime(news_counts['datetime'])

    # Combine both DataFrames
    combined = pd.concat([
        df[['Time']].assign(source='main').rename(columns={'Time': 'time'}),
        news_counts[['datetime']].assign(source='news').rename(columns={'datetime': 'time'})
    ])

    # Sort by time and source
    combined = combined.sort_values(by=['time', 'source'], ascending=[True, False])

    # Find the next and previous news events
    result = combined.copy()
    result['prev_news_time'] = np.where(result['source'] == 'news', result['time'], None)
    result['next_news_time'] = np.where(result['source'] == 'news', result['time'], None)
    
    result['prev_news_time'] = result['prev_news_time'].ffill()
    result['next_news_time'] = result['next_news_time'].bfill()

    # Filter for main events and calculate time differences
    result = result[result['source'] == 'main'].copy()
    
    result['time_week_nbr'] = result['time'].dt.isocalendar().week
    result['prev_time_week_nbr'] = result['prev_news_time'].dt.isocalendar().week
    result['next_time_week_nbr'] = result['next_news_time'].dt.isocalendar().week

    # Calculate seconds since last news event
    result['seconds_since_last_news_event'] = np.where(
        result['time'].dt.isocalendar().week != result['prev_news_time'].dt.isocalendar().week,
        (result['prev_news_time'] - result['time']).dt.total_seconds() + (2 * 86400),
        (result['prev_news_time'] - result['time']).dt.total_seconds()
    )

    # Calculate seconds to next news event
    result['seconds_to_next_news_event'] = np.where(
        result['time'].dt.isocalendar().week != result['next_news_time'].dt.isocalendar().week,
        (result['next_news_time'] - result['time']).dt.total_seconds() - (2 * 86400),
        (result['next_news_time'] - result['time']).dt.total_seconds()
    )

    # Join back to original DataFrame
    final_df = df.merge(
        result[['time', 'seconds_since_last_news_event', 'seconds_to_next_news_event']],
        left_on='Time',
        right_on='time',
        how='left'
    ).drop(columns=['time'])

    return final_df


In [10]:
news_events_gr = pd.read_csv(
    f"{path_to_trading_folder}data_files/calendar_df_full_updated.csv", 
    parse_dates=['datetime']
).groupby('datetime').count()
news_events_gr.loc[:, 'event'] = (news_events_gr['Id'] >= 1).astype(int)
news_events_gr_id_drp = news_events_gr.drop(columns='Id')
# add timezone "UTC" but don't change the times
news_events_gr_id_drp.index = news_events_gr_id_drp.index.tz_localize('UTC', ambiguous='infer')
news_events_gr_id_drp = news_events_gr_id_drp.reset_index()
news_events_gr_id_drp.tail()

,datetime,event
3176,2024-08-05 17:00:00+00:00,1
3177,2024-08-06 12:00:00+00:00,1
3178,2024-08-13 15:30:00+00:00,1
3179,2024-08-14 12:00:00+00:00,1
3180,2024-08-14 15:30:00+00:00,1


In [28]:
full_data_w_news = find_seconds_to_next_news(full_data.reset_index(), news_events_gr_id_drp)
full_data_w_news.loc[:, 'Week_Nbr'] = (
    full_data_w_news['Time'].dt.year.astype(str) + 
    full_data_w_news['Time'].dt.isocalendar().week.astype(str)
)
# print(full_data_w_news.columns)
strategy = TradingGeekStrategy(full_data_w_news.set_index('Time'))
trade_signals = strategy.calculate_trade_management()
trade_signals

,Open,High,Low,Close,signal,stop_loss,take_profit,risk_reward_ratio
Time,,,,,,,,
2024-01-02 00:00:00+00:00,1.10427,1.10433,1.10423,1.10431,0,NaN,NaN,NaN
2024-01-02 00:15:00+00:00,1.10430,1.10434,1.10420,1.10430,0,NaN,NaN,NaN
2024-01-02 00:30:00+00:00,1.10431,1.10437,1.10431,1.10433,0,NaN,NaN,NaN
2024-01-02 00:45:00+00:00,1.10426,1.10447,1.10423,1.10438,0,NaN,NaN,NaN
2024-01-02 01:00:00+00:00,1.10446,1.10446,1.10383,1.10383,0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2024-08-09 22:45:00+00:00,1.09183,1.09189,1.09179,1.09187,0,NaN,NaN,NaN
2024-08-09 23:00:00+00:00,1.09186,1.09194,1.09167,1.09168,0,NaN,NaN,NaN
2024-08-09 23:15:00+00:00,1.09168,1.09179,1.09167,1.09175,0,NaN,NaN,NaN


In [29]:
len(trade_signals[trade_signals['signal'] != 0])

0

In [203]:
def objective(params):
    strategy = SDMomentumStrategy(
        bb_period=int(params['bb_period']),
        bb_std=params['bb_std'],
        keltner_period=int(params['keltner_period']),
        keltner_atr_multiplier=params['keltner_atr_multiplier'],
        rsi_period=int(params['rsi_period']),
        stoch_period=int(params['stoch_period']),
        roc_period=int(params['roc_period'])
    )

    full_data.columns = full_data.columns.str.lower()
    full_data_index_reset = full_data.reset_index()
    full_data_w_strategy = strategy.apply_strategy(full_data_index_reset)
    full_data_w_strategy_news = find_seconds_to_next_news(full_data_w_strategy, 
                                                          news_events_gr_id_drp)
    full_data_w_strategy_news.loc[:, 'Week_Nbr'] = (
        full_data_w_strategy_news['Time'].dt.year.astype(str) + 
        full_data_w_strategy_news['Time'].dt.isocalendar().week.astype(str)
    )
    total_wks = len(full_data_w_strategy_news.Week_Nbr.unique())

    sim = MtSimulator(
        unit='USD',
        balance=400_000.,
        leverage=100.,
        stop_out_level=0.2,
        hedge=True,
        symbols_filename=FOREX_DATA_PATH
    )
    open_orders = {}
    for _, row in full_data_w_strategy_news.iterrows():    
        # check if any of the stop_losses have been hit, or if any of the exit
        # conditions have been met for any of the open trades if not have the 
        # stoploss trail behind the others
        sim.current_time = row['Time']
        long_exit_boolean = (
            (row['close'] >= row['bb_upper']) |  # Price hits upper Bollinger Band
            (row['rsi'] > 70) |  # Overbought on RSI
            (row['stoch_k'] > 80) |  # Overbought on Stochastic
            (row['roc'] < -2) |  # Momentum reversal
            (row['close'] < row['kc_middle']) | # Price below Keltner middle line
            (row['Time'].time() >= time(23, 0, 0)) # Negative Swap protection
        )
        short_exit_boolean = (
            (row['close'] <= row['bb_lower']) |  # Price hits lower Bollinger Band
            (row['rsi'] < 30) |  # Oversold on RSI
            (row['stoch_k'] < 20) |  # Oversold on Stochastic
            (row['roc'] > 2) |  # Momentum reversal
            (row['close'] > row['kc_middle']) | # Price above Keltner middle line
            ((row['Time'].weekday() == 4) & 
            (row['Time'].time() >= time(23, 0, 0))) # Weekend hold protection
        )
        orders_to_remove = []
        for order in open_orders:
            # if the exit conditions have been met, or the stoploss has been hit
            # or the opposite signal has been hit, close the order
            if order.type == OrderType.Buy:
                if ((long_exit_boolean) or (open_orders[order] >= row['low']) or 
                (row['signal'] < 0)): 
                    sim.close_order(order)
                    orders_to_remove.append(order)
                # otherwise update the trailing stoploss
                else:
                    open_orders[order] = max(
                        open_orders[order], row['close'] + row['stop_distance'])
                    
            elif order.type == OrderType.Sell:
                if ((short_exit_boolean) or (open_orders[order] <= row['high'])
                    or (row['signal'] > 0)):
                    sim.close_order(order)
                    orders_to_remove.append(order)
                else:
                    open_orders[order] = min(
                        open_orders[order], row['close'] - row['stop_distance'])
        for o in orders_to_remove:
            open_orders.pop(o)

        if ((row['signal'] > 0) and (row['seconds_to_next_news_event'] > 900) and 
            (row['seconds_since_last_news_event'] < -900) and 
            ((row['Time'].time() >= time(10, 0, 0)) and (row['Time'].time() <= time(17, 0, 0))) and
            (row['Time'].time() < time(23, 0, 0))):
            long_order = sim.create_order(
                order_type=OrderType.Buy,
                symbol='EURUSD',
                volume=row['position_size'],
                fee=max(0., np.random.normal(0.0001, 0.00003)),
            )
            long_stop_loss = long_order.entry_price - row['stop_distance']
            open_orders[long_order] = long_stop_loss


        if ((row['signal'] < 0) and (row['seconds_to_next_news_event'] > 900) and 
            (row['seconds_since_last_news_event'] < -900) and 
            ((row['Time'].time() >= time(10, 0, 0)) and (row['Time'].time() <= time(17, 0, 0))) and
            (((row['Time'].time() < time(23, 0, 0)) and (row['Time'].weekday() == 4)) or 
            row['Time'].weekday() != 4)): 
            short_order = sim.create_order(
                order_type=OrderType.Sell,
                symbol='EURUSD',
                volume=row['position_size'],
                fee=max(0., np.random.normal(0.0001, 0.00003)),
            )
            short_stop_loss = short_order.entry_price + row['stop_distance']
            open_orders[short_order] = short_stop_loss
    state = sim.get_state()
    if len(state['orders']) > 0:
        reward = state['orders']['Profit'].sum()
    else:
        reward = 0

    print(
        f"balance: {state['balance']}, profit: {reward}, total orders: {len(state['orders'])}"
    )

    reward *= -1 
    if (len(state['orders']) < total_wks):
        reward += float('inf')


    return {'loss': reward, 'status': STATUS_OK, 'eval_time': datetime.now(), 'parameters': params} 
    


In [201]:
from hyperopt import hp

search_space = {
    'bb_period': hp.quniform('bb_period', 10, 50, 1),
    'bb_std': hp.uniform('bb_std', 1.5, 3.0),
    'keltner_period': hp.quniform('keltner_period', 10, 50, 1),
    'keltner_atr_multiplier': hp.uniform('keltner_atr_multiplier', 1.0, 2.5),
    'rsi_period': hp.quniform('rsi_period', 7, 21, 1),
    'stoch_period': hp.quniform('stoch_period', 7, 21, 1),
    'roc_period': hp.quniform('roc_period', 5, 20, 1)
}


In [ ]:
trials = Trials()
best = fmin(fn=objective,
            space=search_space,
            algo=tpe.suggest,
            max_evals=100_000, # Number of evaluations of the objective function
            trials=trials,
            trials_save_file=f'{path_to_trading_folder}Indicator_backtesting/Indicator_search/trials_SDMomentum_{datetime.now().strftime("%Y%m%d_%H%M%S")}.pkl')

print("Best parameters:", best)

In [176]:
positions = state['orders']
positions.loc[:, "Week_Nbr"] = positions['Entry Time'].dt.year.astype(str) + positions['Entry Time'].dt.isocalendar().week.astype(str)
len(positions.Week_Nbr.unique())

382

In [ ]:
positions = state['orders']
positions.loc[:, "Week_Nbr"] = positions['Entry Time'].dt.year.astype(str) + positions['Entry Time'].dt.isocalendar().week.astype(str)
# get the average number of positions per week by getting the distinct count of Week_Nbr column
avg_positions_per_week = positions.groupby('Week_Nbr').size().mean()
print(avg_positions_per_week)
positions

In [48]:
# convert the columns of full_data to lowercase
# full_data.columns = full_data.columns.str.lower()
# full_data = full_data.reset_index(drop=True)
# print_trade_example(full_data, SDMomentumStrategy())
signals = run_strategy_example(full_data)
signals

,open,high,low,close,bb_middle,bb_upper,bb_lower,bb_width,kc_middle,kc_upper,...,roc,volatility,vol_ma,high_volatility,squeeze,signal,atr,position_size,stop_distance,stop_level
31,1.11831,1.12086,1.11816,1.11998,1.117123,1.118842,1.115403,0.003078,1.117077,1.118526,...,0.326964,0.000860,NaN,False,False,1,0.00508,1.0,0.007620,1.112360
104,1.11389,1.11512,1.11378,1.11501,1.113505,1.114631,1.112379,0.002023,1.113488,1.114601,...,0.196797,0.000563,NaN,False,False,1,0.00254,1.0,0.003810,1.111200
106,1.11416,1.11494,1.11404,1.11456,1.113531,1.114721,1.112341,0.002138,1.113518,1.114597,...,0.121271,0.000595,NaN,False,False,1,0.00260,1.0,0.003900,1.110660
133,1.10866,1.10877,1.10736,1.10738,1.108881,1.110330,1.107432,0.002613,1.108912,1.110321,...,-0.264789,0.000724,0.001589,False,False,-1,0.00313,1.0,0.004695,1.112075
158,1.11011,1.11225,1.11000,1.11103,1.108498,1.110251,1.106745,0.003163,1.108484,1.109973,...,0.247228,0.000876,0.001381,False,False,1,0.00520,1.0,0.007800,1.103230
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49998,1.09519,1.09549,1.09448,1.09465,1.095847,1.098182,1.093512,0.004261,1.095826,1.098160,...,-0.111327,0.001167,0.001866,False,False,-1,0.00206,1.0,0.003090,1.097740
50020,1.09163,1.09174,1.09117,1.09157,1.092267,1.093914,1.090619,0.003016,1.092239,1.093905,...,-0.093356,0.000824,0.002015,False,False,-1,0.00239,1.0,0.003585,1.095155
50026,1.09187,1.09195,1.09085,1.09134,1.092283,1.093744,1.090822,0.002674,1.092249,1.093643,...,-0.114407,0.000730,0.002018,False,False,-1,0.00268,1.0,0.004020,1.095360
50031,1.09190,1.09340,1.09188,1.09318,1.092146,1.093530,1.090762,0.002535,1.092097,1.093524,...,0.185124,0.000692,0.001992,False,False,1,0.00287,1.0,0.004305,1.088875
